# PyTorch and Automatic Differentiation: Worked Solutions


In [ ]:
import torch
import numpy as np
gen = torch.Generator().manual_seed(23)
dtype = torch.float64


## Problem 1. A reverse product
**Problem.** For $f(x_1,x_2)=(x_1x_2,x_1^2+x_2)$, compute the reverse derivative seeded by a fixed output vector $v$.

**Solution.** Numerator layout gives $J_f=\begin{pmatrix}x_2&x_1\\2x_1&1\end{pmatrix}$. The row derivative of $v^Tf$ is $v^TJ_f$. PyTorch stores its transposed entries $J_f^Tv$ in the input shape. A tensor of shape $(2,)$ has no row/column orientation. The tests check two seeds and several points without constructing the Jacobian in the differentiated function.


In [ ]:
def reverse_product(x, v):
    y = torch.stack((x[0]*x[1], x[0].square()+x[1]))
    (g,) = torch.autograd.grad(y, x, grad_outputs=v)
    return g


In [ ]:
x = torch.tensor([2., 3.], dtype=dtype, requires_grad=True)
v = torch.tensor([4., -1.], dtype=dtype)
g = reverse_product(x, v)
J = torch.tensor([[3., 2.], [4., 1.]], dtype=dtype)
torch.testing.assert_close(g, J.T @ v)
assert g.shape == x.shape
assert x.grad is None
for point in [[-1., 2.], [0., -3.]]:
    x = torch.tensor(point, dtype=dtype, requires_grad=True)
    J = torch.stack((torch.stack((x[1], x[0])),
                     torch.stack((2*x[0], torch.ones((), dtype=dtype)))))
    torch.testing.assert_close(reverse_product(x, v), J.T @ v)


In [ ]:
x = torch.tensor([2., 3.], dtype=dtype, requires_grad=True)
v = torch.tensor([2., -1.], dtype=dtype)
torch.testing.assert_close(reverse_product(x, v), torch.tensor([2., 3.], dtype=dtype))


## Problem 2. Batched affine gradients
**Problem.** Derive parameter gradients for $Z=XW^T+\mathbf1_Bb^T$ with $X:(B,d)$ and $W:(m,d)$, using the entrywise-mean squared error $L=\|Z-T\|_F^2/(Bm)$.

**Solution.** Write $\Delta=2(Z-T)/(Bm)$. Then $\nabla_WL=\Delta^TX$ and $\nabla_bL=\Delta^T\mathbf1_B$. The bias is shared by all rows, so its gradient sums their contributions. All stored parameter gradients have the shape of their parameters. If the objective instead divides only by $B$, both the loss and all gradients are multiplied by $m$.


In [ ]:
def affine_gradients(X, W, b, target):
    Z = X @ W.T + b
    delta = 2*(Z-target)/Z.numel()
    return delta.T @ X, delta.sum(dim=0)


In [ ]:
for B, d, m in [(4, 3, 2), (1, 2, 3)]:
    X = torch.randn(B, d, generator=gen, dtype=dtype)
    W = torch.randn(m, d, generator=gen, dtype=dtype, requires_grad=True)
    b = torch.randn(m, generator=gen, dtype=dtype, requires_grad=True)
    target = torch.randn(B, m, generator=gen, dtype=dtype)
    gW, gb = affine_gradients(X, W.detach(), b.detach(), target)
    loss = (X @ W.T + b - target).square().mean()
    expected_W, expected_b = torch.autograd.grad(loss, (W, b))
    torch.testing.assert_close(gW, expected_W)
    torch.testing.assert_close(gb, expected_b)
    assert gW.shape == W.shape and gb.shape == b.shape


## Problem 3. One gradient step
**Problem.** Compute a detached update for $L(w)=\|w-t\|^2/2$, preserving the original input, and compare several step sizes.

**Solution.** Since $\nabla L(w)=w-t$, the update is $w^+=(1-\eta)w+\eta t$. The error satisfies $w^+-t=(1-\eta)(w-t)$. Therefore $0<\eta<2$ contracts the error; $\eta=1$ reaches the minimum in one step. At $\eta=2.1$ its norm grows by a factor $1.1$ per step and its sign alternates.


In [ ]:
def quadratic_step(w, target, learning_rate):
    leaf = w.detach().clone().requires_grad_(True)
    loss = 0.5*(leaf-target).square().sum()
    (g,) = torch.autograd.grad(loss, leaf)
    with torch.no_grad():
        updated = leaf-learning_rate*g
    return updated


In [ ]:
w = torch.tensor([3., -2.], dtype=dtype)
target = torch.tensor([1., 1.], dtype=dtype)
updated = quadratic_step(w, target, 0.25)
torch.testing.assert_close(updated, 0.75*w+0.25*target)
assert not updated.requires_grad
torch.testing.assert_close(w, torch.tensor([3., -2.], dtype=dtype))
# Controlled experiment: vary only the step size on a fixed quadratic.
for eta in [0.1, 1., 2.1]:
    z = w.clone()
    errors = []
    for _ in range(8):
        z = quadratic_step(z, target, eta)
        errors.append(float(torch.linalg.vector_norm(z-target)))
    print(eta, errors)


## Problem 4. Detached paths, accumulation, and module mode
**Problem.** Explain the discrepancy when the linear branch of $f(x)=x^2+3x$ is detached. Check gradient accumulation and distinguish module evaluation mode from graph recording.

**Solution.** The full derivative is $2x+3$, equal to $7$ at $x=2$. Detachment removes the second path, so autodiff returns $4$. Finite differences of the original numerical function perturb both paths and return approximately $7$. Two calls to `backward` on fresh square graphs add $4+4$ to the leaf gradient; resetting it clears that accumulation. An affine layer's `eval()` does not disable differentiation; `no_grad()` suppresses recording for its operations.


In [ ]:
# A detached branch omits part of the derivative.
x = torch.tensor(2., dtype=dtype, requires_grad=True)
f = x.square() + 3*x.detach()
(g_wrong,) = torch.autograd.grad(f, x)
h = 1e-5
f_numeric = lambda a: a*a+3*a
finite_difference = (f_numeric(2+h)-f_numeric(2-h))/(2*h)
assert abs(finite_difference-7) < 1e-8
assert abs(float(g_wrong)-4) < 1e-12
assert abs(float(g_wrong)-finite_difference) > 2.9
# Accumulation uses fresh forward graphs.
x = torch.tensor(2., dtype=dtype, requires_grad=True)
x.square().backward()
x.square().backward()
assert float(x.grad) == 8.
x.grad = None
x.square().backward()
assert float(x.grad) == 4.
# nn.Linear stores output-by-input weights.
layer = torch.nn.Linear(3, 2, dtype=dtype)
X = torch.ones(4, 3, dtype=dtype)
torch.testing.assert_close(layer(X), X @ layer.weight.T + layer.bias)
layer.eval()
assert layer(X).requires_grad  # evaluation mode does not disable autograd
with torch.no_grad():
    assert not layer(X).requires_grad


## Problem 5. A two-layer composition
**Problem.** Implement a two-layer tanh computation, derive both weight gradients, and compare them with autograd and directional finite differences. State the loss reduction.

**Solution.** Let $X:(B,d)$, $W_1:(h,d)$, $b_1:(h,)$, $W_2:(m,h)$, and $b_2:(m,)$. Define
$$A=XW_1^T+\mathbf1_Bb_1^T,\qquad H=\tanh A,\qquad Z=HW_2^T+\mathbf1_Bb_2^T,$$
$$L=\frac{1}{2Bm}\|Z-T\|_F^2.$$
Set $\Delta_Z=(Z-T)/(Bm)$ and $\Delta_A=(\Delta_ZW_2)\odot(1-H^2)$. Applying the chain rule gives
$$\nabla_{W_2}L=\Delta_Z^TH,\quad \nabla_{b_2}L=\Delta_Z^T\mathbf1_B,\quad \nabla_{W_1}L=\Delta_A^TX,\quad \nabla_{b_1}L=\Delta_A^T\mathbf1_B.$$
For a joint parameter direction $(U_1,u_1,U_2,u_2)$, the directional derivative is the sum of Frobenius inner products of these gradients with the corresponding directions. The centered finite difference perturbs all four parameter blocks simultaneously.


In [ ]:
B, d, h, m = 5, 3, 4, 2
X = torch.randn(B, d, generator=gen, dtype=dtype)
T = torch.randn(B, m, generator=gen, dtype=dtype)
parameters = tuple(torch.randn(*shape, generator=gen, dtype=dtype,
                               requires_grad=True)
                   for shape in [(h, d), (h,), (m, h), (m,)])
W1, b1, W2, b2 = parameters

def objective(W1, b1, W2, b2):
    H = torch.tanh(X @ W1.T + b1)
    Z = H @ W2.T + b2
    return (Z-T).square().sum()/(2*B*m)

with torch.no_grad():
    H = torch.tanh(X @ W1.T + b1)
    Z = H @ W2.T + b2
    delta_Z = (Z-T)/(B*m)
    delta_A = (delta_Z @ W2)*(1-H.square())
    manual = (delta_A.T @ X, delta_A.sum(0),
              delta_Z.T @ H, delta_Z.sum(0))
automatic = torch.autograd.grad(objective(*parameters), parameters)
for actual, expected in zip(automatic, manual):
    torch.testing.assert_close(actual, expected)
directions = tuple(torch.randn(p.shape, generator=gen, dtype=dtype)
                   for p in parameters)
directional = sum((g*u).sum() for g, u in zip(manual, directions))
step = 1e-5
with torch.no_grad():
    plus = objective(*(p+step*u for p, u in zip(parameters, directions)))
    minus = objective(*(p-step*u for p, u in zip(parameters, directions)))
    finite_difference = (plus-minus)/(2*step)
torch.testing.assert_close(finite_difference, directional, rtol=1e-7, atol=1e-8)
print("Directional derivative:", float(directional),
      "finite difference:", float(finite_difference))


## Problem 6. Restoring model state
**Problem.** Save and restore a module's state dictionary and compare outputs on fixed inputs. What is missing for exact training resumption?

**Solution.** Registered parameters and persistent buffers are saved. Loading them into the same architecture reproduces this deterministic CPU computation. Architecture definitions, module mode flags, optimizer and scheduler state, generator states, data-order position, and pending gradient accumulation are not automatically included. Exact continuation also depends on deterministic execution and the numerical environment. The following in-memory serialization avoids creating a file.


In [ ]:
import io
torch.manual_seed(23)
model = torch.nn.Sequential(torch.nn.Linear(3, 4, dtype=dtype),
                           torch.nn.Tanh(), torch.nn.Linear(4, 2, dtype=dtype))
model.eval()
fixed_inputs = torch.tensor([[1., 2., 3.], [-1., 0., 2.]], dtype=dtype)
with torch.no_grad():
    expected = model(fixed_inputs)
buffer = io.BytesIO()
torch.save(model.state_dict(), buffer)
buffer.seek(0)
restored = torch.nn.Sequential(torch.nn.Linear(3, 4, dtype=dtype),
                              torch.nn.Tanh(), torch.nn.Linear(4, 2, dtype=dtype))
restored.load_state_dict(torch.load(buffer, weights_only=True))
restored.eval()
with torch.no_grad():
    torch.testing.assert_close(restored(fixed_inputs), expected, rtol=0, atol=0)
